In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.auto import tqdm
import random

# =====================================================
# Device
# =====================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================
# SpecAugment
# =====================================================

class SpecAugment(nn.Module):
    def __init__(self, freq_mask_param=4, time_mask_param=8):
        super().__init__()
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param

    def forward(self, x):

        if not self.training:
            return x

        x = x.clone()

        B, C, Freq, Time = x.shape

        for i in range(B):

            f = random.randint(0, self.freq_mask_param)
            f0 = random.randint(0, max(0, Freq - f))
            x[i, :, f0:f0+f, :] = 0

            t = random.randint(0, self.time_mask_param)
            t0 = random.randint(0, max(0, Time - t))
            x[i, :, :, t0:t0+t] = 0

        return x

# =====================================================
# Teacher Model
# =====================================================

class KWSCNNTeacherI(nn.Module):
    def __init__(self, num_classes=12):
        super().__init__()

        self.specaug = SpecAugment()

        self.features = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.Conv2d(16,16,3,padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout(0.10),

            nn.Conv2d(16,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout(0.15),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Dropout(0.20),

            nn.Conv2d(128,192,3,padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU()
        )

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Sequential(
            nn.Linear(192,128),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(128,num_classes)
        )

    def forward(self,x):

        x = self.specaug(x)

        feat = self.features(x)

        pooled = self.pool(feat)
        pooled = pooled.flatten(1)

        logits = self.classifier(pooled)

        return logits, feat

# =====================================================
# Student Model
# =====================================================

class BestKWSStudentKD(nn.Module):
    def __init__(self, input_shape, num_classes=12, dropout_prob=0.2818):
        super().__init__()

        self.conv1 = nn.Conv2d(1,8,kernel_size=(40,3),padding=(0,1))
        self.pool1 = nn.MaxPool2d((1,2))

        self.conv2 = nn.Conv2d(8,32,kernel_size=(1,3),padding=(0,1))

        self.dropout = nn.Dropout(dropout_prob)

        # Projection layer for feature alignment
        self.proj = nn.Conv2d(32,192,kernel_size=1)

        with torch.no_grad():
            dummy = torch.zeros(1,*input_shape)

            x = F.relu(self.conv1(dummy))
            x = self.pool1(x)
            x = F.relu(self.conv2(x))

            fc_input = x.numel()

        self.fc = nn.Linear(fc_input,num_classes)

    def forward(self,x):

        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        feat = F.relu(self.conv2(x))

        x = self.dropout(feat)

        flat = torch.flatten(x,1)

        logits = self.fc(flat)

        return logits, feat

# =====================================================
# Load Data
# =====================================================

data = torch.load("/content/drive/MyDrive/gsc_mfcc_40x98.pt")

X_train = data["X_train"].unsqueeze(1).float()
y_train = data["y_train"]

X_val = data["X_val"].unsqueeze(1).float()
y_val = data["y_val"]

X_test = data["X_test"].unsqueeze(1).float()
y_test = data["y_test"]

# =====================================================
# Normalize
# =====================================================

mean = X_train.mean()
std = X_train.std()

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

# =====================================================
# Dataset
# =====================================================

train_ds = torch.utils.data.TensorDataset(X_train, X_train, y_train)
val_ds = torch.utils.data.TensorDataset(X_val, X_val, y_val)
test_ds = torch.utils.data.TensorDataset(X_test, y_test)

train_loader = torch.utils.data.DataLoader(train_ds,batch_size=32,shuffle=True)
val_loader = torch.utils.data.DataLoader(val_ds,batch_size=32)
test_loader = torch.utils.data.DataLoader(test_ds,batch_size=32)

# =====================================================
# Models
# =====================================================

teacher = KWSCNNTeacherI().to(device)
teacher.load_state_dict(torch.load("/content/best_teacher-imp-3-4-s.pth", map_location=device))
teacher.eval()

for p in teacher.parameters():
    p.requires_grad = False

student = BestKWSStudentKD(input_shape=X_train.shape[1:]).to(device)

# =====================================================
# Optimizer
# =====================================================

optimizer = optim.Adam(student.parameters(), lr=3e-4, weight_decay=1e-4)

# =====================================================
# Combined KD Loss
# =====================================================

def total_kd_loss(student_logits, teacher_logits,
                  student_feat, teacher_feat,
                  labels,
                  T=1,
                  alpha=0.3,
                  beta=0.1):

    ce = F.cross_entropy(student_logits, labels)

    kd = F.kl_div(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1),
        reduction='batchmean'
    ) * (T*T)

    # Feature projection
    student_feat_proj = student.proj(student_feat)

    # Spatial alignment if needed
    teacher_feat_resized = F.adaptive_avg_pool2d(
        teacher_feat,
        student_feat_proj.shape[-2:]
    )

    feat_loss = F.mse_loss(student_feat_proj, teacher_feat_resized)

    return alpha*ce + (1-alpha)*kd + beta*feat_loss

# =====================================================
# Training
# =====================================================

best_val_acc = 0

for epoch in range(350):

    student.train()

    running_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for xs, xt, y in pbar:

        xs = xs.to(device)
        xt = xt.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        with torch.no_grad():
            teacher_logits, teacher_feat = teacher(xt)

        student_logits, student_feat = student(xs)

        loss = total_kd_loss(
            student_logits,
            teacher_logits,
            student_feat,
            teacher_feat,
            y
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = student_logits.argmax(1)

        correct += (preds == y).sum().item()
        total += y.size(0)

        pbar.set_postfix(acc=correct/total)

    # =====================================================
    # Validation
    # =====================================================

    student.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for xs, xt, y in val_loader:

            xs = xs.to(device)
            xt = xt.to(device)
            y = y.to(device)

            teacher_logits, teacher_feat = teacher(xt)
            student_logits, student_feat = student(xs)

            preds = student_logits.argmax(1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    val_acc = correct / total

    print(f"\nEpoch {epoch+1} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(student.state_dict(), "best_student_feature_kd.pth")
        print("✅ Best saved")

# =====================================================
# Test
# =====================================================

student.load_state_dict(torch.load("best_student_feature_kd.pth"))
student.eval()

correct = 0
total = 0

with torch.no_grad():

    for x,y in tqdm(test_loader):

        x = x.to(device)
        y = y.to(device)

        logits,_ = student(x)

        preds = logits.argmax(1)

        correct += (preds == y).sum().item()
        total += y.size(0)

print(f"\n✅ Test Accuracy: {correct/total:.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.auto import tqdm
import random

# =====================================================
# Device
# =====================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================
# SpecAugment
# =====================================================

class SpecAugment(nn.Module):
    def __init__(self, freq_mask_param=4, time_mask_param=8):
        super().__init__()
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param

    def forward(self, x):

        if not self.training:
            return x

        x = x.clone()

        B, C, Freq, Time = x.shape

        for i in range(B):

            f = random.randint(0, self.freq_mask_param)
            f0 = random.randint(0, max(0, Freq - f))
            x[i, :, f0:f0+f, :] = 0

            t = random.randint(0, self.time_mask_param)
            t0 = random.randint(0, max(0, Time - t))
            x[i, :, :, t0:t0+t] = 0

        return x

# =====================================================
# Teacher Model
# =====================================================

class KWSCNNTeacherI(nn.Module):
    def __init__(self, num_classes=12):
        super().__init__()

        self.specaug = SpecAugment()

        self.features = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.Conv2d(16,16,3,padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout(0.10),

            nn.Conv2d(16,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout(0.15),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Dropout(0.20),

            nn.Conv2d(128,192,3,padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU()
        )

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Sequential(
            nn.Linear(192,128),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(128,num_classes)
        )

    def forward(self,x):

        x = self.specaug(x)

        feat = self.features(x)

        pooled = self.pool(feat)
        pooled = pooled.flatten(1)

        logits = self.classifier(pooled)

        return logits, feat

# =====================================================
# Student Model
# =====================================================

class BestKWSStudentKD(nn.Module):
    def __init__(self, input_shape, num_classes=12, dropout_prob=0.2818):
        super().__init__()

        self.conv1 = nn.Conv2d(1,8,kernel_size=(40,3),padding=(0,1))
        self.pool1 = nn.MaxPool2d((1,2))

        self.conv2 = nn.Conv2d(8,32,kernel_size=(1,3),padding=(0,1))

        self.dropout = nn.Dropout(dropout_prob)

        # Projection layer for feature alignment
        self.proj = nn.Conv2d(32,192,kernel_size=1)

        with torch.no_grad():
            dummy = torch.zeros(1,*input_shape)

            x = F.relu(self.conv1(dummy))
            x = self.pool1(x)
            x = F.relu(self.conv2(x))

            fc_input = x.numel()

        self.fc = nn.Linear(fc_input,num_classes)

    def forward(self,x):

        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        feat = F.relu(self.conv2(x))

        x = self.dropout(feat)

        flat = torch.flatten(x,1)

        logits = self.fc(flat)

        return logits, feat

# =====================================================
# Load Data
# =====================================================

data = torch.load("/content/drive/MyDrive/gsc_mfcc_40x98.pt")

X_train = data["X_train"].unsqueeze(1).float()
y_train = data["y_train"]

X_val = data["X_val"].unsqueeze(1).float()
y_val = data["y_val"]

X_test = data["X_test"].unsqueeze(1).float()
y_test = data["y_test"]

# =====================================================
# Normalize
# =====================================================

mean = X_train.mean()
std = X_train.std()

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

# =====================================================
# Dataset
# =====================================================

train_ds = torch.utils.data.TensorDataset(X_train, X_train, y_train)
val_ds = torch.utils.data.TensorDataset(X_val, X_val, y_val)
test_ds = torch.utils.data.TensorDataset(X_test, y_test)

train_loader = torch.utils.data.DataLoader(train_ds,batch_size=32,shuffle=True)
val_loader = torch.utils.data.DataLoader(val_ds,batch_size=32)
test_loader = torch.utils.data.DataLoader(test_ds,batch_size=32)

# =====================================================
# Models
# =====================================================

teacher = KWSCNNTeacherI().to(device)
teacher.load_state_dict(torch.load("/content/best_teacher-imp-3-4-s.pth", map_location=device))
teacher.eval()

for p in teacher.parameters():
    p.requires_grad = False

student = BestKWSStudentKD(input_shape=X_train.shape[1:]).to(device)

# =====================================================
# Optimizer
# =====================================================

optimizer = optim.Adam(student.parameters(), lr=3e-4, weight_decay=1e-4)

# =====================================================
# Combined KD Loss
# =====================================================

def total_kd_loss(student_logits, teacher_logits,
                  student_feat, teacher_feat,
                  labels,
                  T=1,
                  alpha=0.7,
                  beta=0.3):

    ce = F.cross_entropy(student_logits, labels)

    kd = F.kl_div(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1),
        reduction='batchmean'
    ) * (T*T)

    # Feature projection
    student_feat_proj = student.proj(student_feat)

    # Spatial alignment if needed
    teacher_feat_resized = F.adaptive_avg_pool2d(
        teacher_feat,
        student_feat_proj.shape[-2:]
    )

    feat_loss = F.mse_loss(student_feat_proj, teacher_feat_resized)

    return alpha*ce + (1-alpha)*kd + beta*feat_loss

# =====================================================
# Training
# =====================================================

best_val_acc = 0

for epoch in range(350):

    student.train()

    running_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for xs, xt, y in pbar:

        xs = xs.to(device)
        xt = xt.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        with torch.no_grad():
            teacher_logits, teacher_feat = teacher(xt)

        student_logits, student_feat = student(xs)

        loss = total_kd_loss(
            student_logits,
            teacher_logits,
            student_feat,
            teacher_feat,
            y
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = student_logits.argmax(1)

        correct += (preds == y).sum().item()
        total += y.size(0)

        pbar.set_postfix(acc=correct/total)

    # =====================================================
    # Validation
    # =====================================================

    student.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for xs, xt, y in val_loader:

            xs = xs.to(device)
            xt = xt.to(device)
            y = y.to(device)

            teacher_logits, teacher_feat = teacher(xt)
            student_logits, student_feat = student(xs)

            preds = student_logits.argmax(1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    val_acc = correct / total

    print(f"\nEpoch {epoch+1} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(student.state_dict(), "best_student_feature_kd.pth")
        print("✅ Best saved")

# =====================================================
# Test
# =====================================================

student.load_state_dict(torch.load("best_student_feature_kd.pth"))
student.eval()

correct = 0
total = 0

with torch.no_grad():

    for x,y in tqdm(test_loader):

        x = x.to(device)
        y = y.to(device)

        logits,_ = student(x)

        preds = logits.argmax(1)

        correct += (preds == y).sum().item()
        total += y.size(0)

print(f"\n✅ Test Accuracy: {correct/total:.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.auto import tqdm
import random

# =====================================================
# Device
# =====================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================
# SpecAugment
# =====================================================

class SpecAugment(nn.Module):
    def __init__(self, freq_mask_param=4, time_mask_param=8):
        super().__init__()
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param

    def forward(self, x):

        if not self.training:
            return x

        x = x.clone()

        B, C, Freq, Time = x.shape

        for i in range(B):

            f = random.randint(0, self.freq_mask_param)
            f0 = random.randint(0, max(0, Freq - f))
            x[i, :, f0:f0+f, :] = 0

            t = random.randint(0, self.time_mask_param)
            t0 = random.randint(0, max(0, Time - t))
            x[i, :, :, t0:t0+t] = 0

        return x

# =====================================================
# Teacher Model
# =====================================================

class KWSCNNTeacherI(nn.Module):
    def __init__(self, num_classes=12):
        super().__init__()

        self.specaug = SpecAugment()

        self.features = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.Conv2d(16,16,3,padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout(0.10),

            nn.Conv2d(16,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout(0.15),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Dropout(0.20),

            nn.Conv2d(128,192,3,padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU()
        )

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Sequential(
            nn.Linear(192,128),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(128,num_classes)
        )

    def forward(self,x):

        x = self.specaug(x)

        feat = self.features(x)

        pooled = self.pool(feat)
        pooled = pooled.flatten(1)

        logits = self.classifier(pooled)

        return logits, feat

# =====================================================
# Student Model
# =====================================================

class BestKWSStudentKD(nn.Module):
    def __init__(self, input_shape, num_classes=12, dropout_prob=0.2818):
        super().__init__()

        self.conv1 = nn.Conv2d(1,8,kernel_size=(40,3),padding=(0,1))
        self.pool1 = nn.MaxPool2d((1,2))

        self.conv2 = nn.Conv2d(8,32,kernel_size=(1,3),padding=(0,1))

        self.dropout = nn.Dropout(dropout_prob)

        # Projection layer for feature alignment
        self.proj = nn.Conv2d(32,192,kernel_size=1)

        with torch.no_grad():
            dummy = torch.zeros(1,*input_shape)

            x = F.relu(self.conv1(dummy))
            x = self.pool1(x)
            x = F.relu(self.conv2(x))

            fc_input = x.numel()

        self.fc = nn.Linear(fc_input,num_classes)

    def forward(self,x):

        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        feat = F.relu(self.conv2(x))

        x = self.dropout(feat)

        flat = torch.flatten(x,1)

        logits = self.fc(flat)

        return logits, feat

# =====================================================
# Load Data
# =====================================================

data = torch.load("/content/drive/MyDrive/gsc_mfcc_40x98.pt")

X_train = data["X_train"].unsqueeze(1).float()
y_train = data["y_train"]

X_val = data["X_val"].unsqueeze(1).float()
y_val = data["y_val"]

X_test = data["X_test"].unsqueeze(1).float()
y_test = data["y_test"]

# =====================================================
# Normalize
# =====================================================

mean = X_train.mean()
std = X_train.std()

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

# =====================================================
# Dataset
# =====================================================

train_ds = torch.utils.data.TensorDataset(X_train, X_train, y_train)
val_ds = torch.utils.data.TensorDataset(X_val, X_val, y_val)
test_ds = torch.utils.data.TensorDataset(X_test, y_test)

train_loader = torch.utils.data.DataLoader(train_ds,batch_size=32,shuffle=True)
val_loader = torch.utils.data.DataLoader(val_ds,batch_size=32)
test_loader = torch.utils.data.DataLoader(test_ds,batch_size=32)

# =====================================================
# Models
# =====================================================

teacher = KWSCNNTeacherI().to(device)
teacher.load_state_dict(torch.load("/content/best_teacher-imp-3-4-s.pth", map_location=device))
teacher.eval()

for p in teacher.parameters():
    p.requires_grad = False

student = BestKWSStudentKD(input_shape=X_train.shape[1:]).to(device)

# =====================================================
# Optimizer
# =====================================================

optimizer = optim.Adam(student.parameters(), lr=3e-4, weight_decay=1e-4)

# =====================================================
# Combined KD Loss
# =====================================================

def total_kd_loss(student_logits, teacher_logits,
                  student_feat, teacher_feat,
                  labels,
                  T=1,
                  alpha=0.15,
                  beta=0.3):

    ce = F.cross_entropy(student_logits, labels)

    kd = F.kl_div(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1),
        reduction='batchmean'
    ) * (T*T)

    # Feature projection
    student_feat_proj = student.proj(student_feat)

    # Spatial alignment if needed
    teacher_feat_resized = F.adaptive_avg_pool2d(
        teacher_feat,
        student_feat_proj.shape[-2:]
    )

    feat_loss = F.mse_loss(student_feat_proj, teacher_feat_resized)

    return alpha*ce + (1-alpha)*kd + beta*feat_loss

# =====================================================
# Training
# =====================================================

best_val_acc = 0

for epoch in range(350):

    student.train()

    running_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for xs, xt, y in pbar:

        xs = xs.to(device)
        xt = xt.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        with torch.no_grad():
            teacher_logits, teacher_feat = teacher(xt)

        student_logits, student_feat = student(xs)

        loss = total_kd_loss(
            student_logits,
            teacher_logits,
            student_feat,
            teacher_feat,
            y
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = student_logits.argmax(1)

        correct += (preds == y).sum().item()
        total += y.size(0)

        pbar.set_postfix(acc=correct/total)

    # =====================================================
    # Validation
    # =====================================================

    student.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for xs, xt, y in val_loader:

            xs = xs.to(device)
            xt = xt.to(device)
            y = y.to(device)

            teacher_logits, teacher_feat = teacher(xt)
            student_logits, student_feat = student(xs)

            preds = student_logits.argmax(1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    val_acc = correct / total

    print(f"\nEpoch {epoch+1} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(student.state_dict(), "best_student_feature_kd.pth")
        print("✅ Best saved")

# =====================================================
# Test
# =====================================================

student.load_state_dict(torch.load("best_student_feature_kd.pth"))
student.eval()

correct = 0
total = 0

with torch.no_grad():

    for x,y in tqdm(test_loader):

        x = x.to(device)
        y = y.to(device)

        logits,_ = student(x)

        preds = logits.argmax(1)

        correct += (preds == y).sum().item()
        total += y.size(0)

print(f"\n✅ Test Accuracy: {correct/total:.4f}")